# 0. Load in the cleaned test and train data

In [127]:
import pandas as pd

test = pd.read_csv("/Users/nick/Library/CloudStorage/OneDrive-Personal/Programming projects/Team Union 2/Team-Union/Nick/churn_test_cleaned.csv")
test.info()
submission = test[["GuestID"]].copy()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1739 entries, 0 to 1738
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   GuestID         1739 non-null   int64 
 1   PromoCode       1739 non-null   object
 2   Region          1739 non-null   object
 3   AllInclusive    1739 non-null   int64 
 4   PackageType     1739 non-null   object
 5   VIP             1739 non-null   int64 
 6   RoomService     1739 non-null   int64 
 7   Dining          1739 non-null   int64 
 8   Retail          1739 non-null   int64 
 9   Spa             1739 non-null   int64 
 10  Entertainment   1739 non-null   int64 
 11  LoyaltyPoints   1739 non-null   int64 
 12  SurveyScore     1739 non-null   int64 
 13  DaysSinceEmail  1739 non-null   int64 
 14  BookingChannel  1739 non-null   object
 15  AgeGroup        1739 non-null   object
 16  ReferralSource  1739 non-null   object
 17  SharedRoom      1739 non-null   int64 
 18  RoomFloo

In [128]:
test.isna().sum()

GuestID           0
PromoCode         0
Region            0
AllInclusive      0
PackageType       0
VIP               0
RoomService       0
Dining            0
Retail            0
Spa               0
Entertainment     0
LoyaltyPoints     0
SurveyScore       0
DaysSinceEmail    0
BookingChannel    0
AgeGroup          0
ReferralSource    0
SharedRoom        0
RoomFloor         0
RoomNumber        0
RoomSide          0
dtype: int64

In [129]:
train = pd.read_csv("/Users/nick/Library/CloudStorage/OneDrive-Personal/Programming projects/Team Union 2/Team-Union/Nick/churn_train_cleaned.csv")
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6954 entries, 0 to 6953
Data columns (total 22 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   GuestID         6954 non-null   int64 
 1   PromoCode       6954 non-null   object
 2   Region          6954 non-null   object
 3   AllInclusive    6954 non-null   int64 
 4   PackageType     6954 non-null   object
 5   VIP             6954 non-null   int64 
 6   RoomService     6954 non-null   int64 
 7   Dining          6954 non-null   int64 
 8   Retail          6954 non-null   int64 
 9   Spa             6954 non-null   int64 
 10  Entertainment   6954 non-null   int64 
 11  LoyaltyPoints   6954 non-null   int64 
 12  SurveyScore     6954 non-null   int64 
 13  DaysSinceEmail  6954 non-null   int64 
 14  BookingChannel  6954 non-null   object
 15  AgeGroup        6954 non-null   object
 16  ReferralSource  6954 non-null   object
 17  Churned         6954 non-null   int64 
 18  SharedRo

In [130]:
train.isna().sum()

GuestID           0
PromoCode         0
Region            0
AllInclusive      0
PackageType       0
VIP               0
RoomService       0
Dining            0
Retail            0
Spa               0
Entertainment     0
LoyaltyPoints     0
SurveyScore       0
DaysSinceEmail    0
BookingChannel    0
AgeGroup          0
ReferralSource    0
Churned           0
SharedRoom        0
RoomFloor         0
RoomNumber        0
RoomSide          0
dtype: int64

In [131]:
#Create X and y variables for modeling

from sklearn.model_selection import train_test_split

X = train.drop('Churned', axis=1)
y = train['Churned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 1. Train the model - This is where your model goes

In [ ]:
import optuna
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.model_selection import StratifiedKFold, train_test_split

# GuestID is an identifier — drop from features.
drop_cols = ["GuestID", "Churned"]
X = train.drop(columns=drop_cols)
y = train["Churned"]

# XGBoost needs numeric features. Build shared category maps from train+test so the
# test-time encoding lines up with what the model was trained on.
cat_features = [c for c in X.columns if X[c].dtype == "object"]
test_features = test.drop(columns=["GuestID"]) if "GuestID" in test.columns else test.copy()

category_maps = {}
for c in cat_features:
    combined = pd.concat([X[c].astype(str), test_features[c].astype(str)]).fillna("missing")
    category_maps[c] = pd.Categorical(combined).categories

X_enc = X.copy()
for c in cat_features:
    X_enc[c] = pd.Categorical(
        X_enc[c].astype(str).fillna("missing"), categories=category_maps[c]
    ).codes

X_train, X_test, y_train, y_test = train_test_split(
    X_enc, y, test_size=0.2, random_state=42, stratify=y
)

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos


def objective(trial):
    params = {
        "n_estimators": 1000,
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "scale_pos_weight": scale_pos_weight,
        "eval_metric": "auc",
        "early_stopping_rounds": 50,
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_train)):
        X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
        y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]

        model = XGBClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        preds = model.predict(X_va)
        scores.append(roc_auc_score(y_va, preds))

        trial.report(sum(scores) / len(scores), fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return sum(scores) / len(scores)


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2),
)
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"Best CV ROC-AUC: {study.best_value:.4f}")
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

# Refit best model on the train split, evaluate on the held-out test split.
best_params = {
    **study.best_params,
    "n_estimators": 1000,
    "scale_pos_weight": scale_pos_weight,
    "eval_metric": "auc",
    "early_stopping_rounds": 50,
    "random_state": 42,
    "n_jobs": -1,
    "tree_method": "hist",
}

xgb = XGBClassifier(**best_params)
xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

y_pred = xgb.predict(X_test)
print(f"\nTest ROC-AUC:   {roc_auc_score(y_test, y_pred):.3f}")
print(f"Best iteration: {xgb.best_iteration}")
print(classification_report(y_test, y_pred))

fi = pd.DataFrame({
    "feature": X_train.columns,
    "importance": xgb.feature_importances_,
}).sort_values("importance", ascending=False)
print(fi.to_string(index=False))

# 2. Test - This will produce the test data

In [133]:
test.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1739 entries, 0 to 1738
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   GuestID         1739 non-null   int64 
 1   PromoCode       1739 non-null   object
 2   Region          1739 non-null   object
 3   AllInclusive    1739 non-null   int64 
 4   PackageType     1739 non-null   object
 5   VIP             1739 non-null   int64 
 6   RoomService     1739 non-null   int64 
 7   Dining          1739 non-null   int64 
 8   Retail          1739 non-null   int64 
 9   Spa             1739 non-null   int64 
 10  Entertainment   1739 non-null   int64 
 11  LoyaltyPoints   1739 non-null   int64 
 12  SurveyScore     1739 non-null   int64 
 13  DaysSinceEmail  1739 non-null   int64 
 14  BookingChannel  1739 non-null   object
 15  AgeGroup        1739 non-null   object
 16  ReferralSource  1739 non-null   object
 17  SharedRoom      1739 non-null   int64 
 18  RoomFloo

In [134]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6954 entries, 0 to 6953
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   PromoCode       6954 non-null   object
 1   Region          6954 non-null   object
 2   AllInclusive    6954 non-null   int64 
 3   PackageType     6954 non-null   object
 4   VIP             6954 non-null   int64 
 5   RoomService     6954 non-null   int64 
 6   Dining          6954 non-null   int64 
 7   Retail          6954 non-null   int64 
 8   Spa             6954 non-null   int64 
 9   Entertainment   6954 non-null   int64 
 10  LoyaltyPoints   6954 non-null   int64 
 11  SurveyScore     6954 non-null   int64 
 12  DaysSinceEmail  6954 non-null   int64 
 13  BookingChannel  6954 non-null   object
 14  AgeGroup        6954 non-null   object
 15  ReferralSource  6954 non-null   object
 16  SharedRoom      6954 non-null   int64 
 17  RoomFloor       6954 non-null   object
 18  RoomNumb

In [ ]:
#run the test data against the optimized XGBoost model to get the predictions for the submission file

#drop GuestID column from the test data
test_X = test.drop(columns=["GuestID"]) if "GuestID" in test.columns else test.copy()

# Apply the same categorical encoding used during training so feature indices line up.
for c in cat_features:
    test_X[c] = pd.Categorical(
        test_X[c].astype(str).fillna("missing"), categories=category_maps[c]
    ).codes

# Ensure column order matches the training feature order
test_X = test_X[X_train.columns]

probs = xgb.predict(test_X)

#display the probs
print(probs)

#submission already has GuestID; attach predictions as "Churned"
submission["Churned"] = probs

submission.info()

#output the csv file with the predictions
submission.to_csv("submission.csv", index=False)